Goal: turn your team's unwritten coding rules into files that steer AI code helpers and keep them within your process — then prove those files are present and well‑formed.

Roadmap (what we do):
- We work in a small sandbox repo, create two Markdown governance docs: one for code suggestions (repo‑wide agent rules) and one for agent personas/workflows.
- We add concrete, checkable guidance: where code lives, how to import, how to test/lint, what to never do, and when to escalate.
- We write a tiny validator script that checks structure, examples, and forbidden patterns and run it locally.
- We commit the results to git (local commit; no push by default).

What you should understand afterward:
- How to encode team norms into precise, reusable instructions that AI helpers follow.
- How to define an agent persona with responsibilities, allowed actions, templates, and boundaries.
- How to wire a lightweight validator you can run locally and in CI to keep these docs from drifting.

Assumes: you can read/write Python and Markdown, and you know basic git commands. Likely gaps we fill: how Python repos are laid out (src/ vs tests/), what a repo‑level AI instruction file controls, how to define agent personas, and how to do simple programmatic Markdown checks.

This is an artifact lesson: every code cell writes or validates a real file and shows concrete results.

Pipeline map — what flows where:

```text
[Start] 
   |
   v
[Create sandbox repo]
   |
   v
[Write .github/copilot-instructions.md]
   |             \
   |              \
   v               v
[Write AGENTS.md]  [Explain structure & examples]
   \               /
    \             /
     v           v
   [Write tools/validate_agent_docs.py]
             |
             v
      [Run validator -> JSON report]
             |
             v
        [Git add + commit]
             |
             v
           [End]
```
Use this as a legend: “copilot-instructions” = repo‑wide rules for AI code helpers; “AGENTS.md” = personas/workflows/boundaries; “validator” = Python script that checks structure/content and emits a JSON report.

Why repository‑level agent instructions exist
- They turn high‑level team norms into concrete, reusable guardrails. Without them, prompts drift and agents guess.
- Example of drift:
  - Ambiguous: "Write a function to parse config and add tests."
  - Precise (repo‑level instruction): "Add functions under src/package_name/, use absolute imports, add pytest tests under tests/, run pytest -q locally, and include type hints; never commit secrets; escalate if touching auth."
This builds on your prompt‑engineering intuition but moves it into a shared, versioned doc the whole team and every agent can follow.

Python repo layout — the minimum a helper must know
- src/: source code lives here; package import root is src/package_name/.
- tests/: tests live here; use pytest discovery.
- Commands that matter: pytest -q (tests), mypy (types), ruff/flake8 (lint), black (format).
- Naming: prefer absolute imports from the package (from package_name.module import X), not relative (from ..module import X). Agents need these anchors to generate code that runs and is importable.

Anatomy of the two docs we will create
- .github/copilot-instructions.md: purpose, scope, coding conventions (imports, typing, tests, lint), examples (Before/After code blocks), security constraints (allowed/forbidden), escalation guidance.
- AGENTS.md: for each agent persona add responsibilities, allowed actions, task templates (with parameters like {{issue_number}}), sample interactions (a request and an agent reply with a code block), and boundaries/sensitive‑data rules.
Examples should be concrete and checkable — include code fences ``` and explicit Before/After labels.

In [ ]:
# Setup check — run me first
import importlib, sys, os, shutil, subprocess, textwrap, json, re
from pathlib import Path

REQUIRED = {"git (CLI)": "Install git and ensure it is on your PATH"}
missing = []
if shutil.which("git") is None:
    missing.append("git (CLI) — Install git and ensure it is on your PATH")

py_ok = tuple(map(int, sys.version.split()[0].split("."))) >= (3, 8)
if not py_ok:
    missing.append("Python >=3.8 (found %s)" % sys.version.split()[0])

if missing:
    raise SystemExit("Missing prerequisites:\n  - " + "\n  - ".join(missing))

print("Setup OK — Python", sys.version.split()[0])
print("git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Create or reuse a sandbox git repo so this lesson is safe to run anywhere
from pathlib import Path
import subprocess, os

SANDBOX = Path.cwd() / "agent-docs-demo-repo"
SANDBOX.mkdir(exist_ok=True)
# Initialize repo if needed
if not (SANDBOX / ".git").exists():
    subprocess.run(["git", "init", "-q"], cwd=SANDBOX)
    # Create minimal src/ and tests/ structure to reference
    (SANDBOX / "src" / "package_name").mkdir(parents=True, exist_ok=True)
    (SANDBOX / "tests").mkdir(parents=True, exist_ok=True)
    # Add a .gitignore to keep repo clean
    (SANDBOX / ".gitignore").write_text("__pycache__/\n*.pyc\n", encoding="utf-8")
    subprocess.run(["git", "add", ".gitignore"], cwd=SANDBOX)
    subprocess.run(["git", "commit", "-m", "chore: init demo repo"], cwd=SANDBOX, capture_output=True)

# Ensure .github/ and tools/ directories exist
(SANDBOX / ".github").mkdir(parents=True, exist_ok=True)
(SANDBOX / "tools").mkdir(parents=True, exist_ok=True)

print("Working in sandbox repo:", str(SANDBOX))
print("Repo status:")
print(subprocess.run(["git", "status", "-s"], cwd=SANDBOX, capture_output=True, text=True).stdout or "(clean)")

We will create files inside agent-docs-demo-repo to avoid modifying your real project while you learn the workflow. Paths will mirror what you will later add to your actual repo: .github/copilot-instructions.md, AGENTS.md, and tools/validate_agent_docs.py.

Writing .github/copilot-instructions.md
Where we are on the map: “Write .github/copilot-instructions.md”.
What matters in this cell:
- We include required sections and concrete examples with Before/After code fences.
- We encode Python‑repo conventions: src/, package_name, tests/, pytest, mypy, ruff, black.
- We state allowed/forbidden actions and escalation triggers.
Note: replace occurrences of package_name with your real package when you adapt this to your project.

In [ ]:
# Write .github/copilot-instructions.md with required structure and examples
from textwrap import dedent

copilot_md = dedent(
    """
    # Repository-wide Copilot Instructions

    ## Purpose
    This document provides clear, actionable guidance for AI code assistants working in this repository. It encodes our coding style, layout, testing, and safety rules so generated code integrates cleanly and safely.

    ## Scope
    These rules apply to code suggestions, refactors, documentation edits, and test additions in this repository. They do not grant permission to access external systems or secrets; where unsure, escalate to a human reviewer.

    ## Coding conventions
    - Layout: put application code under `src/package_name/` and tests under `tests/`.
    - Imports: prefer absolute imports from the package (e.g., `from package_name.utils import parse_cfg`) over relative imports.
    - Typing: include Python type hints; prefer `typing` and `collections.abc` types. Run `mypy` locally before proposing large changes.
    - Tests: for new or changed behavior, add or update `pytest` tests under `tests/` and run `pytest -q`.
    - Lint & format: follow `ruff` (or flake8) and `black`. Keep functions small and names descriptive.
    - CI hooks: assume CI will run `pytest -q`, `ruff`, `black --check`, and `mypy`.

    ## Examples
    Before: relative import without typing or tests

    ```python
    # src/package_name/handlers.py
    from ..utils import slugify

    def build_url(name):
        return f"/items/{slugify(name)}"
    ```

    After: absolute import with typing and a matching test

    ```python
    # src/package_name/handlers.py
    from package_name.utils import slugify

    def build_url(name: str) -> str:
        return f"/items/{slugify(name)}"
    ```

    ```python
    # tests/test_handlers.py
    from package_name.handlers import build_url

    def test_build_url_simple():
        assert build_url("Thing") == "/items/thing"
    ```

    ## Security constraints
    - Never include secrets, tokens, or credentials in code, tests, or docs.
    - Do not suggest shelling out with `ssh -i`, invoking `curl http://internal` endpoints, or embedding keys (e.g., AWS access keys) in examples.
    - Avoid adding network calls unless the task explicitly requires it and an approved mock/stub exists.
    - Prefer local mocks and fakes for external services.

    ## Escalation
    Escalate to a human reviewer when tasks affect: authentication/authorization, database migrations, production configuration, or any area with access to sensitive data. When escalating, propose a short plan and a checklist (tests, lint, type checks) rather than code.
    """
).lstrip()

repo = SANDBOX
path = repo / ".github" / "copilot-instructions.md"
path.write_text(copilot_md, encoding="utf-8")
print("Wrote:", path)
print("Size:", path.stat().st_size, "bytes")

Created .github/copilot-instructions.md with all required sections and example code blocks. When you adapt this for a real repo, replace package_name with your actual package import root and verify the test/lint commands match your tooling.

Writing AGENTS.md
Where we are on the map: “Write AGENTS.md”.
What matters in this cell:
- Define at least one agent with the subheadings: Responsibilities, Allowed actions, Task templates, Sample interactions, Boundaries and sensitive‑data rules.
- Include parameterized placeholders like {{issue_number}} in templates.
- Include at least one sample interaction with a request and an agent reply that contains a code block.

In [ ]:
# Write AGENTS.md with personas, workflows, and boundaries
agents_md = dedent(
    """
    # AGENTS — Personas and Operating Rules

    This document defines agent roles, allowed actions, task templates, sample interactions, and boundaries so AI helpers follow our process consistently.

    ## Agent: Code Writer — Persona

    ### Responsibilities
    - Implement features and bug fixes under `src/package_name/`.
    - Create or update `pytest` tests under `tests/` for changed behavior.
    - Keep code typed and lint‑clean (mypy, ruff, black --check).

    ### Allowed actions
    - Create/modify files under `src/package_name/` and `tests/`.
    - Propose CLI commands to run locally: `pytest -q`, `mypy`, `ruff`, `black --check`.
    - Add small utility functions where needed, with tests.

    ### Task templates
    - Implement function {{function_name}} in module {{module_path}} with signature {{signature}} and add tests in `tests/test_{{module_basename}}.py`.
    - Fix bug described in issue #{{issue_number}} by reproducing with a failing test, then patching code in {{module_path}}.

    ### Sample interactions
    Request:

    > Add a function to normalize product titles and test it.

    Agent reply:

    ```python
    # src/package_name/text.py
    import re
    from typing import Iterable

    def normalize_title(title: str) -> str:
        """Lowercase, collapse whitespace, and strip punctuation."""
        title = re.sub(r"\s+", " ", title.strip()).lower()
        return re.sub(r"[!?,.:]", "", title)
    ```

    ```python
    # tests/test_text.py
    from package_name.text import normalize_title

    def test_normalize_title_basic():
        assert normalize_title(" Hello, World!  ") == "hello world"
    ```

    ### Boundaries and sensitive-data rules
    - Do not introduce network access or external API calls unless a stub/mock exists and the task requires it.
    - Never include secrets, passwords, or tokens in code, tests, or docs.
    - Escalate when tasks touch authentication, PII, or migrations.

    ## Agent: Reviewer — Persona

    ### Responsibilities
    - Review diffs for correctness, safety, and adherence to conventions.
    - Ensure tests and type checks pass locally.

    ### Allowed actions
    - Comment on pull requests with actionable feedback and checklists.
    - Request changes or approvals; may amend trivial style issues.

    ### Task templates
    - Review PR #{{pr_number}} focusing on {{focus_area}}; verify `pytest -q`, `mypy`, and linters pass; ensure changes stay within scope.

    ### Sample interactions
    Request:

    > Review PR #123 that refactors utils.

    Agent reply:

    ```text
    Checklist:\n- [ ] Unit tests pass (pytest -q)\n- [ ] Types pass (mypy)\n- [ ] Lint/format clean (ruff, black --check)\n- [ ] Absolute imports maintained; no behavior changes without tests
    ```

    ### Boundaries and sensitive-data rules
    - Do not approve changes that add secrets, bypass tests, or alter auth flows without escalation.
    - Escalate to a human owner for risky or ambiguous changes.
    """
).lstrip()

agents_path = repo / "AGENTS.md"
agents_path.write_text(agents_md, encoding="utf-8")
print("Wrote:", agents_path)
print("Size:", agents_path.stat().st_size, "bytes")

Created AGENTS.md at the repo root with two agents. Each agent block includes the required subheadings, parameterized task templates ({{...}}), and a sample interaction that shows a request and a reply with a code block.

Validation design — what we check and why
- Existence: both files must be present.
- Required headings: we look for case‑insensitive matches of the required sections.
- Examples/code fences: at least one triple‑backtick code block and a Before/After example in copilot-instructions.md.
- Repository tokens: mention of src/, tests/, pytest, or mypy so conventions are concrete.
- Forbidden patterns: scan for risky strings like AWS access keys (AKIA...), literal PASSWORD, ssh -i, and curl http://internal. We keep PASSWORD detection case‑sensitive to reduce false positives in prose (documented below).
- Section length: each required section must have at least MIN_SECTION_CHARS characters of content (default 50). Rationale: this nudges authors to add a few sentences, not one‑liners. Adjust via the MIN_SECTION_CHARS environment variable if your team prefers shorter/longer sections.

In [ ]:
# Write tools/validate_agent_docs.py — emits JSON_REPORT=... and a human summary
validator_py = dedent(
    """
    #!/usr/bin/env python3
    import json, os, re, sys
    from pathlib import Path
    from typing import Dict, List, Tuple

    REPO = Path(__file__).resolve().parents[1]
    MIN_SECTION_CHARS = int(os.getenv("MIN_SECTION_CHARS", "50"))

    def read(p: Path) -> str:
        try:
            return p.read_text(encoding="utf-8")
        except FileNotFoundError:
            return ""

    def find_headings(md: str) -> List[Tuple[str, int]]:
        # Return list of (heading_text_lower, start_index)
        result = []
        for m in re.finditer(r"^(#{2,6})\s*(.+?)\s*$", md, flags=re.MULTILINE):
            result.append((m.group(2).strip().lower(), m.start()))
        return result

    def section_lengths(md: str, names: List[str]) -> Dict[str, int]:
        headings = find_headings(md)
        idx_by_name = {h: i for i, (h, _) in enumerate(headings)}
        lengths = {}
        for name in names:
            name_l = name.lower()
            # Find the first heading that case-insensitively equals the name
            match_idx = None
            for i, (h, _) in enumerate(headings):
                if h == name_l:
                    match_idx = i
                    break
            if match_idx is None:
                lengths[name] = 0
                continue
            start = headings[match_idx][1]
            end = headings[match_idx + 1][1] if match_idx + 1 < len(headings) else len(md)
            content = md[start:end]
            # Remove the heading line itself
            content = re.sub(r"^#{2,6}.*$\n?", "", content, count=1, flags=re.MULTILINE)
            lengths[name] = len(content.strip())
        return lengths

    def check_copilot_instructions(p: Path) -> Dict:
        md = read(p)
        ok = True
        messages: List[str] = []
        if not md:
            return {"ok": False, "messages": [f"Missing file: {p}"]}
        req = [
            "Purpose",
            "Scope",
            "Coding conventions",
            "Examples",
            "Security constraints",
            "Escalation",
        ]
        headings = {h for h, _ in find_headings(md)}
        for r in req:
            if r.lower() not in headings:
                ok = False
                messages.append(f"Missing required heading: {r}")
        # Section length checks
        lengths = section_lengths(md, req)
        for name, L in lengths.items():
            if L < MIN_SECTION_CHARS:
                ok = False
                messages.append(f"Section '{name}' too short: {L} chars (min {MIN_SECTION_CHARS})")
        # Example and code fences
        code_fences = md.count("```")
        if code_fences < 2:
            ok = False
            messages.append("Expected at least one fenced code block (```)")
        if ("Before:" not in md) or ("After:" not in md):
            ok = False
            messages.append("Expected a Before:/After: example pair")
        # Repo tokens present
        if not any(tok in md for tok in ("src/", "tests/", "pytest", "mypy")):
            ok = False
            messages.append("Expected mention of at least one of: src/, tests/, pytest, mypy")
        # Forbidden patterns
        forbidden = [
            (r"AKIA[0-9A-Z]{16}", "AWS access key pattern"),
            (r"ssh\s+-i\b", "ssh -i private key usage"),
            (r"curl\s+http://internal\b", "internal curl HTTP endpoint"),
            # Case-sensitive 'PASSWORD' to reduce false positives in general prose
            (r"PASSWORD", "literal PASSWORD token"),
        ]
        for pat, desc in forbidden:
            if re.search(pat, md):
                ok = False
                messages.append(f"Forbidden pattern present ({desc})")
        return {"ok": ok, "messages": messages, "lengths": lengths}

    def slice_between(md: str, start_i: int, end_i: int) -> str:
        return md[start_i:end_i]

    def check_agents_md(p: Path) -> Dict:
        md = read(p)
        ok = True
        messages: List[str] = []
        if not md:
            return {"ok": False, "messages": [f"Missing file: {p}"]}
        # Identify agent blocks by '## ' headings that include 'agent' or '— persona'
        blocks = []
        all_heads = list(re.finditer(r"^(##|###)\s*(.+?)\s*$", md, flags=re.MULTILINE))
        for i, m in enumerate(all_heads):
            text = m.group(2).strip().lower()
            if ("agent" in text) or ("persona" in text):
                start = m.end()
                end = all_heads[i + 1].start() if i + 1 < len(all_heads) else len(md)
                blocks.append((m.group(2).strip(), md[start:end]))
        if not blocks:
            return {"ok": False, "messages": ["No agent blocks found (## Agent: ...)"]}
        # Required subheadings per agent (as ### ...)
        req_subs = [
            "Responsibilities",
            "Allowed actions",
            "Task templates",
            "Sample interactions",
            "Boundaries and sensitive-data rules",
        ]
        agent_results = []
        any_ok = False
        for title, body in blocks:
            subs_present = set(h.lower() for h in re.findall(r"^###\s*(.+?)\s*$", body, flags=re.MULTILINE))
            missing = [s for s in req_subs if s.lower() not in subs_present]
            body_ok = len(missing) == 0
            # Template placeholder check
            tmpl_ok = bool(re.search(r"\{\{[^}]+\}\}", body))
            if not tmpl_ok:
                body_ok = False
                missing.append("parameterized template ({{...}})")
            # Sample interactions: request + reply + at least one code block fence
            sample_sec = re.split(r"^###\s*Sample interactions\s*$", body, maxsplit=1, flags=re.MULTILINE)
            sample_ok = False
            if len(sample_sec) > 1:
                sample_body = sample_sec[1]
                req_present = re.search(r"Request:\s*", sample_body)
                rep_present = re.search(r"Agent reply:\s*", sample_body)
                code_fence = sample_body.count("```") >= 1
                sample_ok = bool(req_present and rep_present and code_fence)
            if not sample_ok:
                body_ok = False
                missing.append("sample interaction with request + agent reply + code block")
            agent_results.append({"title": title, "ok": body_ok, "missing": missing})
            any_ok = any_ok or body_ok
        # All agents should pass; but we report details for each
        if not all(ar["ok"] for ar in agent_results):
            ok = False
            for ar in agent_results:
                if not ar["ok"]:
                    messages.append(f"Agent '{ar['title']}' missing: {', '.join(ar['missing'])}")
        return {"ok": ok, "messages": messages, "agents": agent_results}

    def main() -> int:
        report = {"ok": True, "files": {}}
        cpath = REPO / ".github" / "copilot-instructions.md"
        apath = REPO / "AGENTS.md"
        c = check_copilot_instructions(cpath)
        a = check_agents_md(apath)
        report["files"][str(cpath)] = c
        report["files"][str(apath)] = a
        report["ok"] = bool(c.get("ok") and a.get("ok"))
        # Human-readable summary
        if report["ok"]:
            print("Validation PASSED for:")
            for fname in report["files"].keys():
                print(" -", fname)
        else:
            print("Validation FAILED:")
            for fname, res in report["files"].items():
                if not res.get("ok"):
                    print(" -", fname)
                    for m in res.get("messages", []):
                        print("    *", m)
        # Emit machine-readable line the notebook will parse
        print("JSON_REPORT=" + json.dumps(report, ensure_ascii=False))
        return 0 if report["ok"] else 1

    if __name__ == "__main__":
        sys.exit(main())
    """
).lstrip()

validator_path = repo / "tools" / "validate_agent_docs.py"
validator_path.write_text(validator_py, encoding="utf-8")
# Make it executable on POSIX (best-effort)
try:
    os.chmod(validator_path, 0o755)
except Exception:
    pass
print("Wrote:", validator_path)
print("Size:", validator_path.stat().st_size, "bytes")

What the validator does
- Checks both files exist and searches for required headings.
- Ensures copilot-instructions.md has at least one code fence and a Before:/After: example, mentions repo tokens (src/, tests/, pytest, mypy), and does not contain forbidden patterns.
- Ensures each agent block in AGENTS.md has the five subheadings, at least one {{...}} template placeholder, and a sample interaction with a request, an agent reply, and a code block.
- Section length minimum is MIN_SECTION_CHARS (default 50). Adjust by setting the MIN_SECTION_CHARS environment variable for stricter/looser policies.
- It prints a human summary and a JSON_REPORT=... line. Our notebook will parse that JSON_REPORT line. Note on PASSWORD detection: it is case‑sensitive by design to reduce false positives where prose mentions “password” generically; tune to your risk appetite.

In [ ]:
# Run the validator and parse JSON_REPORT — prints stdout/stderr for debugging if needed
import subprocess, sys, json

proc = subprocess.run([sys.executable, str(validator_path)], cwd=repo, capture_output=True, text=True)
print("--- validator stdout ---\n" + proc.stdout)
if proc.stderr:
    print("--- validator stderr ---\n" + proc.stderr)

# Parse JSON_REPORT line
report = None
for line in proc.stdout.splitlines():
    if line.startswith("JSON_REPORT="):
        payload = line.split("JSON_REPORT=", 1)[1]
        try:
            report = json.loads(payload)
        except json.JSONDecodeError as e:
            print("Failed to parse JSON_REPORT:", e)
            print("Payload was:\n", payload)
        break

if report is None:
    raise SystemExit("Validator did not emit JSON_REPORT=... — cannot parse structured result. Inspect stdout/stderr above.")

# Friendly assertion with remediation hint
if proc.returncode != 0 or not report.get("ok"):
    print("Validator returned non-zero or report not ok. Suggestions:")
    print("- Open the files listed under 'FAILED' above and add missing sections/length/examples.")
    print("- If your sections are intentionally brief, lower MIN_SECTION_CHARS (e.g., set env var to 20) and rerun.")
    raise SystemExit("Validation failed — fix issues and re-run.")

# Print a compact success summary
passed = [k for k, v in report["files"].items() if v.get("ok")]
print("Validation OK for", len(passed), "files.")

Interpreting the output
- The stdout section shows a pass/fail summary for each file.
- On pass, you should see both file paths listed under “Validation PASSED”.
- The notebook parsed the JSON_REPORT line to double‑check and printed a compact success summary. If it ever fails, the stdout/stderr prints above give you immediate clues and suggested fixes.

Commit the artifacts locally
Where we are on the map: “Git add + commit”.
Brief: we stage the three files and create a local commit. To avoid global config surprises in the sandbox, we set a local git user.name/email only if unset. We do not push by default.

In [ ]:
# Stage and commit; set local git identity if needed
import subprocess, shlex

def _git(args: str):
    return subprocess.run(shlex.split(args), cwd=repo, capture_output=True, text=True)

# Ensure local identity in sandbox (don't touch global)
email = _git("git config user.email").stdout.strip()
name = _git("git config user.name").stdout.strip()
if not email:
    _git("git config user.email devnull@example.com")
if not name:
    _git("git config user.name Demo User")

# Add files
add = _git("git add .github/copilot-instructions.md AGENTS.md tools/validate_agent_docs.py")
if add.returncode != 0:
    print(add.stderr)
    raise SystemExit("git add failed")

# Commit
msg = "docs: add agent governance docs and validator"
commit = _git(f"git commit -m {shlex.quote(msg)}")
print(commit.stdout or commit.stderr)

# Show commit hash
rev = _git("git rev-parse HEAD")
print("Committed revision:", rev.stdout.strip())

Local commit created. You can inspect the diff with git show or open the files in the sandbox folder. Pushing to a remote is intentionally not automated here; run git push from your real repo/branch when you adapt this workflow.

Optional CI integration (illustrative — not run here)
Add a GitHub Actions workflow so PRs fail if these docs regress:

```yaml
name: Validate agent docs
on: [push, pull_request]
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - run: python tools/validate_agent_docs.py
```
Tune MIN_SECTION_CHARS by setting env: { MIN_SECTION_CHARS: '80' } if you want stricter section lengths.

Takeaways
- Repository‑level AI instructions remove ambiguity: they encode where code lives, how to import, how to test/lint, what is forbidden, and when to escalate.
- AGENTS.md defines personas and boundaries so helpers act within your process; include responsibilities, allowed actions, templates with parameters, and realistic sample interactions.
- A small validator catches omissions and risky content early; emit both human and machine‑readable output so it works locally and in CI.
- Keep thresholds documented and adjustable (e.g., MIN_SECTION_CHARS). Be explicit about forbidden patterns and note any trade‑offs (like case sensitivity) to reduce surprise.
- Workflow: edit docs → run validator → iterate → commit → (optionally) wire CI to keep the bar consistent on every PR.